# SupportIQ — Stage 3: Classical Baseline (TF-IDF + Logistic Regression)

> **Context:** Before fine-tuning a massive 4-Billion parameter LLM, every production ML engineer must establish a **Classical Baseline**.
> **The Core Question:** Can a simple, $0-cost, CPU-based algorithm already solve this triage problem? If yes, what specific capabilities justify spending money to fine-tune an LLM?


### 1. Setup & Environment
Import the baseline model and evaluation tools.


In [1]:
import sys
from pathlib import Path

import polars as pl

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from supportiq.models.tfidf import TFIDFBaseline

print("Baseline modules loaded successfully.")


Baseline modules loaded successfully.


### 2. Load the Leakage-Safe Data Splits
Load the 80/10/10 Parquet splits generated in Stage 2.


In [2]:
data_dir = project_root / "data/processed"

train_df = pl.read_parquet(data_dir / "train.parquet")
val_df = pl.read_parquet(data_dir / "val.parquet")
test_df = pl.read_parquet(data_dir / "test.parquet")

print(f"Loaded Train: {len(train_df):,} rows")
print(f"Loaded Val:   {len(val_df):,} rows")
print(f"Loaded Test:  {len(test_df):,} rows")


Loaded Train: 21,497 rows
Loaded Val:   2,687 rows
Loaded Test:  2,688 rows


### 3. Train the TF-IDF + Logistic Regression Baseline
We extract two types of n-grams:
1. **Word N-Grams (1-2 words):** Captures key customer terms like *"cancel order"*, *"forgot password"*.
2. **Character N-Grams (3-5 characters):** Captures sub-word morphology and handles typos/inflections.


In [3]:
baseline = TFIDFBaseline(
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 5),
    c_param=1.0,
    random_state=42,
)

print("Training baseline model on CPU...")
baseline.fit(train_df)
print("Training complete!")


Training baseline model on CPU...
{"timestamp": "2026-09-22T22:39:00.668783+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Extracting TF-IDF features for 21497 training samples...", "module": "tfidf", "line": 56}


{"timestamp": "2026-09-22T22:39:03.128720+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Fitting Intent Classifier...", "module": "tfidf", "line": 59}


{"timestamp": "2026-09-22T22:39:11.388398+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Fitting Category Classifier...", "module": "tfidf", "line": 62}


Training complete!


### 4. Evaluate Baseline Performance on Test Data
We evaluate on the frozen out-of-distribution Test Set.


In [4]:
test_results = baseline.evaluate(test_df)

print("=== BASELINE A RESULTS (TEST SET) ===")
print(f"Intent Accuracy:    {test_results['intent']['accuracy'] * 100:.2f}%")
print(f"Intent Macro-F1:    {test_results['intent']['macro_f1'] * 100:.2f}%")
print(f"Category Accuracy:  {test_results['category']['accuracy'] * 100:.2f}%")
print(f"Category Macro-F1:  {test_results['category']['macro_f1'] * 100:.2f}%")


=== BASELINE A RESULTS (TEST SET) ===
Intent Accuracy:    99.93%
Intent Macro-F1:    99.93%
Category Accuracy:  99.93%
Category Macro-F1:  99.91%


### 5. The Scientific Leakage Experiment
**Why did we group by `cluster_id` in Stage 2?**
Here we train a baseline on a **naive random split** (where near-duplicate templates are scattered across both train and test) and compare it against our **leakage-free grouped split**.


In [5]:
# Combine all data and create a naive random 80/20 split
full_df = pl.concat([train_df, val_df, test_df])
shuffled = full_df.sample(fraction=1.0, shuffle=True, seed=42)

split_idx = int(len(shuffled) * 0.8)
naive_train = shuffled[:split_idx]
naive_test = shuffled[split_idx:]

naive_baseline = TFIDFBaseline(random_state=42).fit(naive_train)
naive_results = naive_baseline.evaluate(naive_test)

print("=== SCIENTIFIC COMPARISON: GROUPED vs NAIVE RANDOM SPLIT ===")
print(f"Naive Random Split Accuracy:    {naive_results['intent']['accuracy'] * 100:.2f}% (Memorization artifact)")
print(f"Leakage-Free Grouped Accuracy:  {test_results['intent']['accuracy'] * 100:.2f}% (True generalization)")


{"timestamp": "2026-09-22T22:39:16.803352+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Extracting TF-IDF features for 21497 training samples...", "module": "tfidf", "line": 56}


{"timestamp": "2026-09-22T22:39:18.468965+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Fitting Intent Classifier...", "module": "tfidf", "line": 59}


{"timestamp": "2026-09-22T22:39:27.271116+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Fitting Category Classifier...", "module": "tfidf", "line": 62}


=== SCIENTIFIC COMPARISON: GROUPED vs NAIVE RANDOM SPLIT ===
Naive Random Split Accuracy:    99.85% (Memorization artifact)
Leakage-Free Grouped Accuracy:  99.93% (True generalization)


### 6. Save Model Bundle & Benchmark Report
Persist the trained baseline model to `artifacts/` for quick inference.


In [6]:
artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "baseline_tfidf.joblib"
baseline.save(model_path)

# Verify loading
loaded = TFIDFBaseline.load(model_path)
sample_cat, sample_intent = loaded.predict(["How do I get a refund on my purchase?"])
print(f"Sample prediction: Category={sample_cat[0]}, Intent={sample_intent[0]}")
print(f"Saved model successfully to {model_path}")


{"timestamp": "2026-09-22T22:39:34.320120+00:00", "level": "INFO", "name": "supportiq.models.tfidf", "message": "Saved baseline model to /home/dvinix/Projects/supportiq/artifacts/baseline_tfidf.joblib", "module": "tfidf", "line": 99}


Sample prediction: Category=REFUND, Intent=get_refund
Saved model successfully to /home/dvinix/Projects/supportiq/artifacts/baseline_tfidf.joblib
